# Convert a heretic model to phone-browser (MLC) format
**Honest note:** I could not run this notebook myself, so it is untested. It follows the official MLC-LLM conversion steps.
Use a **free Colab GPU** (Runtime > Change runtime type > T4 GPU). It takes ~10-20 minutes.

What it does: downloads `p-e-w/Qwen3-4B-Instruct-2507-heretic` (an uncensored Qwen3 4B), converts it to 4-bit `q4f16_1` MLC format, and uploads it to YOUR Hugging Face account. Then you paste the resulting address into the app (Settings > Models > Add your own model, built on **Qwen3-4B**).

You need a free Hugging Face account and a **write** token: huggingface.co/settings/tokens

In [ ]:
!pip install -q --pre -U -f https://mlc.ai/wheels mlc-llm-nightly-cu122 mlc-ai-nightly-cu122
!pip install -q -U huggingface_hub
!git lfs install
import mlc_llm; print("mlc_llm OK")

In [ ]:
SOURCE = "p-e-w/Qwen3-4B-Instruct-2507-heretic"   # model to convert
OFFICIAL = "mlc-ai/Qwen3-4B-q4f16_1-MLC"          # official build with the same architecture (for the chat template)
MY_REPO_NAME = "Qwen3-4B-heretic-q4f16_1-MLC"     # name of the new repo on your account
QUANT = "q4f16_1"

In [ ]:
from huggingface_hub import snapshot_download
src = snapshot_download(SOURCE, local_dir="src_model")
off = snapshot_download(OFFICIAL, local_dir="official", allow_patterns=["mlc-chat-config.json"])
print(src)

In [ ]:
!mlc_llm convert_weight src_model --quantization {QUANT} -o out

In [ ]:
import json
off_cfg = json.load(open("official/mlc-chat-config.json"))
tpl = off_cfg["conversation"]  # reuse the official chat template exactly
!mlc_llm gen_config src_model --quantization {QUANT} --conv-template qwen2 --context-window-size 4096 -o out
cfg = json.load(open("out/mlc-chat-config.json"))
cfg["conversation"] = tpl
cfg["context_window_size"] = 4096
json.dump(cfg, open("out/mlc-chat-config.json","w"), indent=1)
print("config ready")

In [ ]:
from huggingface_hub import login, HfApi
login()   # paste your WRITE token
api = HfApi(); me = api.whoami()["name"]
repo = f"{me}/{MY_REPO_NAME}"
api.create_repo(repo, exist_ok=True)
api.upload_folder(folder_path="out", repo_id=repo)
print("DONE. Paste this into the app:\nhttps://huggingface.co/" + repo)

### If something fails
- `conv-template` error: run `!mlc_llm gen_config --help` and pick the Qwen template it lists.
- Out of memory on Colab: restart the runtime and run only the cells in order.
- It converted but the phone says it cannot load: the model may be too big for Safari's GPU limits. Try the 1.7B or a 3B model instead.